# 05: Comprehensive Validation Report

Generates a single-file HTML report from **measured** values only. The endpoint comparison uses `pipeline.fitting.compare_tier2_vs_golden` (real LTspice runs). Metrics that have not been measured here (switching-loss error, wall-clock speedup) are deliberately omitted — the report never invents a value.

In [ ]:
import sys, os
from pathlib import Path
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import yaml
from common.types import CurveData
from config.yaml_io import load_output
from pipeline.fitting import fit_pwl_model, compare_tier2_vs_golden
from reporting.html_report import generate_html_report

VENDOR = 'irfs7730pbf'   # any config/inputs/<name>.yaml stem
cfg = yaml.safe_load(Path(f'config/inputs/{VENDOR}.yaml').read_text(encoding='utf-8'))
print(f'Vendor: {VENDOR}')

In [ ]:
lib = fit_pwl_model(VENDOR)
rows, worst_pct = compare_tier2_vs_golden(VENDOR, lib)
print(f'Worst endpoint error: {worst_pct:.2f}%')

c1 = load_output(VENDOR, 'curve1_output_characteristics')['extracted']
vmax = float(cfg['conditions']['Vds_sweep_max'])
prefix = 'curve_id_vgs_'
keys = sorted((k for k in c1 if k.startswith(prefix)),
              key=lambda k: float(k[len(prefix):-2]))
curves = []
for k in keys:
    y = c1[k]
    vgs = float(k[len(prefix):-2])
    curves.append(CurveData(
        name=f'Vgs = {vgs:g} V',
        x_name='Drain-Source Voltage Vds', x_unit='V',
        y_name='Drain Current Id', y_unit='A',
        x_data=np.linspace(0, vmax, len(y)).tolist(), y_data=list(y),
        family_param='Vgs', family_value=vgs))

out = Path('reports/validation_report.html')
out.parent.mkdir(parents=True, exist_ok=True)
# Only the measured endpoint error is supplied; loss error and speedup
# are omitted because this notebook does not measure them.
generate_html_report(VENDOR, out, curves,
                     metrics={'endpoint_error_pct': round(worst_pct, 3)})
print(f'HTML report: {out.resolve()}')